# Generate Training Data
## Specification Parameter

In [ ]:
import numpy as np
import random

alpha = 0.5
beta = 0.3
epsilon = 1.5
delta = 2

## Define Trace
Genete a sine function as a trace and augment it with noise

In [ ]:
start = 0           # Start of the interval (in radians)
end = 6 * np.pi     # End of the interval (one full sine wave cycle)

def generate_trace(length):
    time = np.linspace(start, end, length)
    scale = random.uniform(1, 50)
    var = random.uniform(1,30)
    # Actual ground truth data
    gt = scale*(np.sin(time) + np.sin(time*var))
    d = delta * random.uniform(-1,1)
    e = np.random.uniform(-1, 1, length)
    m = gt + epsilon * e + d
    return m

## Define Monitor

In [ ]:
# takes z_n and a measurement and produces z_(n+1)
def monitor(z, i):
    [s,b] = z[:,0]
    G = z[:,1:]
    e_s = G[0,:]
    e_b = G[1,:]

    # Compute new center
    sn = alpha * i + (1-alpha) * (s + b)
    bn = beta * (sn - s) + (1 - beta) * b

    # New Slack Variables
    e_s_n = np.zeros(len(e_s)+1)
    e_s_n[-1] = alpha*epsilon
    e_s_n[0] = alpha*delta

    # Extend Old State
    e_s = np.append(e_s, 0)
    e_b = np.append(e_b, 0)

    # Compute new Support
    e_s_n = e_s_n + (1-alpha) * (e_s + e_b)
    e_b_n = beta * (e_s_n - e_s) + (1-beta) * e_b

    Gn = np.vstack([e_s_n, e_b_n])
    Cn = np.vstack([sn, bn])

    return np.hstack([Cn, Gn])

# Takes the first measurement and returns z0
def init_state(i0):
    s = np.array([i0])
    b = np.array([0])
    e_s = np.zeros(2)
    e_s[0] = delta
    e_s[1] = epsilon
    e_b = np.zeros(2)
    Gn = np.vstack([e_s, e_b])
    Cn = np.vstack([s, b])
    return np.hstack([Cn, Gn])

## Generate Zonotopes

In [ ]:
# Pads an existing zonope with zero valued support vectors until it has exatly size many
def pad_zonotope(z, size):
    [r,s] = z.shape
    pad = np.zeros((r,(size-s)+2))
    return np.hstack([z, pad])

def generate_zonotopes(length):
    trace = generate_trace(length)
    zonotopes = [init_state(trace[0])]
    for i in range(1,10):
        zonotopes.append(monitor(zonotopes[-1],trace[i]))
    padded = np.array([pad_zonotope(z, length) for z in zonotopes])
    return(trace, padded)

[trace, gt] = generate_zonotopes(10)
generate_zonotopes(10)
print(gt.shape)

(10, 2, 12)
